# Jupyter + PyVista 计算模拟三维可视化

本 notebook 是后续主线示例：用 Python 生成一个三维标量场，把它保存成 VTK 时间序列，并在 Jupyter 中完成切片、等值面和截图导出。ParaView 只作为可选工具；生成的 `.pvd/.vti` 文件仍可在桌面版 ParaView 中打开。

## 1. 环境与依赖

先使用课程环境启动 JupyterLab：

```bash
conda activate ai4math-vis
jupyter lab simulation_visualization_jupyter_pyvista.ipynb
```

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv

from simulation_visualization_demo import (
    configure_off_screen,
    make_image_data,
    make_coordinates,
    scalar_field,
    write_time_series,
)

print("PyVista", pv.__version__)
print("VTK", pv.vtk_version_info)

## 2. 生成模拟数据

这里用一个随时间变化的三维标量场模拟 Poisson/热传导类结果。真实项目中，只需要把这一步替换为读取求解器输出的 NumPy 数组、HDF5 或已有 VTK 文件。

In [ ]:
output_dir = Path("outputs/simulation_demo")
files, scalar_range = write_time_series(output_dir, n=48, steps=12)

grid = pv.read(files[0])
print(f"VTK files: {len(files)}")
print("PVD:", output_dir / "poisson_demo.pvd")
print("scalar range:", scalar_range)
grid

## 3. Notebook 交互后端

优先使用 `trame` 后端。如果当前浏览器或远程环境不支持交互视图，后面的静态截图单元仍然可以运行。

In [ ]:
try:
    pv.set_jupyter_backend("trame")
    print("Using PyVista Jupyter backend: trame")
except Exception as exc:
    print("Could not enable trame backend:", exc)
    print("Static/off-screen rendering cells can still be used.")

## 4. 正交切片

正交切片适合查看三维标量场内部结构。颜色范围固定为整个时间序列的最小值和最大值，便于不同时间步之间比较。

In [ ]:
slices = grid.slice_orthogonal(x=0.0, y=0.0, z=0.0)

p = pv.Plotter(window_size=(900, 650))
p.add_mesh(
    slices,
    scalars="potential",
    cmap="coolwarm",
    clim=scalar_range,
    scalar_bar_args={"title": "potential"},
)
p.add_axes(line_width=2)
p.show_grid(color="black", grid="back", location="outer")
p.camera_position = [(2.6, -3.1, 2.2), (0.0, 0.0, 0.0), (0.0, 0.0, 1.0)]
p.show()

## 5. 等值面

等值面展示满足 `potential = c` 的三维结构，适合观察源项、界面、涡量或误差阈值区域。

In [ ]:
contour = grid.contour(isosurfaces=[-0.20, 0.20], scalars="potential")

p = pv.Plotter(window_size=(900, 650))
p.add_mesh(
    contour,
    scalars="potential",
    cmap="coolwarm",
    clim=scalar_range,
    smooth_shading=True,
    scalar_bar_args={"title": "potential"},
)
p.add_mesh(grid.outline(), color="black", line_width=1)
p.add_axes(line_width=2)
p.show_grid(color="black", grid="back", location="outer")
p.camera_position = [(2.6, -3.1, 2.2), (0.0, 0.0, 0.0), (0.0, 0.0, 1.0)]
p.show()

## 6. 向量 glyph

示例数据同时保存了 `minus_gradient` 向量场。实际流体模拟中，这一步可以替换成速度场 `velocity`。先抽样再画箭头，避免三维图过密。

In [ ]:
sampled = grid.extract_subset((0, grid.dimensions[0] - 1, 4, 0, grid.dimensions[1] - 1, 4, 0, grid.dimensions[2] - 1, 4))
arrows = sampled.glyph(orient="minus_gradient", scale="minus_gradient", factor=0.045)

p = pv.Plotter(window_size=(900, 650))
p.add_mesh(grid.slice(normal="z", origin=(0, 0, 0)), scalars="potential", cmap="coolwarm", clim=scalar_range, opacity=0.75)
p.add_mesh(arrows, color="black")
p.add_mesh(grid.outline(), color="black", line_width=1)
p.add_axes(line_width=2)
p.camera_position = [(2.6, -3.1, 2.2), (0.0, 0.0, 0.0), (0.0, 0.0, 1.0)]
p.show()

## 7. 离屏导出报告图片

交互查看用于探索，最终报告图建议固定相机、色标和分辨率后导出。

In [ ]:
configure_off_screen()

figure_path = output_dir / "jupyter_slice.png"
p = pv.Plotter(off_screen=True, window_size=(1200, 850))
p.add_mesh(slices, scalars="potential", cmap="coolwarm", clim=scalar_range, scalar_bar_args={"title": "potential"})
p.add_axes(line_width=2)
p.show_grid(color="black", grid="back", location="outer")
p.camera_position = [(2.6, -3.1, 2.2), (0.0, 0.0, 0.0), (0.0, 0.0, 1.0)]
p.screenshot(figure_path)
p.close()

figure_path